In [8]:
import pickle
import os
from models.tokenizer import AsmTokenizer
import torch
from models.dataset import BERTDataset, BERTANPDataset
from models.bert import BERT2
from torch.utils.data import DataLoader
import numpy as np
import random
tokenizer = AsmTokenizer(vocab_file=os.path.join('.', "outputs", f"baseline-vocab.txt"))
binary = os.path.join('.', 'baseline', 'clamav', 'output_x64-clang-3.5-O0_clambc.pkl')
with open(binary, 'rb') as f:
    data = pickle.load(f)
data['optget'].keys()

Vocab loaded from .\outputs\baseline-vocab.txt


dict_keys([4202576, 4202726, 4202606, 4202650, 4202623, 4202709, 4202668, 4202639, 4202592, 4202685, 4202696, 4202734, 'adjacency_matrix'])

In [ ]:
instr_blocks = []
for key in data['optget'].keys():
    if key != 'adjacency_matrix':
        block_text =' '.join(data['optget'][key])
        instr_blocks.append(block_text)
adj = data['optget']['adjacency_matrix']
mlm_dataset = BERTDataset(instr_blocks, tokenizer, max_len=128)
mlm_loader = DataLoader(mlm_dataset, batch_size=32, shuffle=True)
model = BERT2(vocab_size=len(tokenizer.vocab), seq_len=128+2, device="cuda").to("cuda")

In [ ]:
for ids, labels in mlm_loader:
    ids = ids.to("cuda")
    labels = labels.to("cuda")
    loss, logits = model.forward_mlm(ids, labels)
    break

In [ ]:
anp_dataset = BERTANPDataset(instr_blocks, tokenizer, adj, max_len=128)
anp_loader = DataLoader(anp_dataset, batch_size=32, shuffle=True)
for a, b, label in anp_loader:
    a = a.to("cuda")
    b = b.to("cuda")
    label = label.to("cuda")
    loss, logits = model.forward_anp(a, b, label)
    break